In [ ]:
%%bash
rm -rf /tmp/dbt-fabric-bundle
tar -xzf /lakehouse/default/Files/onelake/pkgs/dbt-fabric-bundle.tar.gz -C /tmp
pip install -q --no-index --find-links=/tmp/dbt-fabric-bundle/wheels dbt-core dbt-fabricspark

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from dbt.cli.main import dbtRunner

PROJECTS = ["dbt-adventureworks", "dbt-jaffle-shop"]
TARGET = "fabric-fabric"

def run_dbt_project(project):
    project_dir = f"/tmp/dbt-fabric-bundle/projects/{project}"
    os.environ["DBT_PROFILES_DIR"] = project_dir
    
    runner = dbtRunner()
    for cmd in [["deps"], ["run"], ["test"]]:
        result = runner.invoke(cmd + ["--project-dir", project_dir, "--target", TARGET])
        print(f"[{project}] {cmd[0]}: {'success' if result.success else 'failed'}")
    return project

with ThreadPoolExecutor(max_workers=2) as executor:
    results = list(executor.map(run_dbt_project, PROJECTS))

We only clean once since the session is shared across both dbt projects:

In [ ]:
import requests, notebookutils, yaml

cfg = yaml.safe_load(open(f"/tmp/dbt-fabric-bundle/projects/{PROJECTS[0]}/profiles.yml"))["adventureworks"]["outputs"][TARGET]
session_id = open(cfg["session_id_file"]).read().strip()

url = f"https://api.fabric.microsoft.com/v1/workspaces/{cfg['workspaceid']}/lakehouses/{cfg['lakehouseid']}/livyApi/versions/2023-12-01/sessions/{session_id}"
r = requests.delete(url, headers={"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"})
print(f"Delete session {session_id}: {r.status_code} {r.reason}")